# 021 — Training (heteroscedastic / NLL)

Train `attention_unet_nll` — the heteroscedastic pilot described in `code-review.md` §7.6 — on the RGB → IR translation task.
Instead of a single deterministic value per pixel, the model predicts a mean `mu` and a log-variance `log_var`, trained with the Gaussian negative-log-likelihood loss (`scripts.losses.gaussian_nll_loss`).

This notebook mirrors `020_training.ipynb` but is entirely separate: it does not modify or retrain `unet`, `resunet`, `attention_unet`, or `efficientnet_unet` — their checkpoints are untouched.

Checkpoint saved to `models/attention_unet_nll/best_model.keras`.
Logs written to `logs/attention_unet_nll/` for TensorBoard.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    grouped_train_val_test_split,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.trainer_nll import compile_model_nll, get_callbacks, get_model_nll
from scripts.visualization import plot_training_curves

# Seed Python / NumPy / TensorFlow from settings.SEED for reproducible runs.
set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Build datasets

In [ ]:
# Artworks split logic: every artwork ID (including the paint-on-support
# mockup groups) is a group kept entirely within a single fold, to prevent
# leakage between sections of the same painting. This can hold an entire
# mockup group out of training even though those groups exist to aid it.
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = grouped_train_val_test_split(
    pairs,
    train_ratio= settings.TRAIN_RATIO,
    val_ratio= settings.VAL_RATIO,
    seed=settings.SEED,
)

# crop_size only affects the augmented training split; val stays full-image.
train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

In [ ]:
# Artwork and mockups split logic (active): real artworks are still
# grouped and leakage-free, but the mockup groups in
# settings.MOCKUP_ARTWORK_IDS are split at the individual pair level, with
# only a small fraction (default 5%) held out for test — the rest is
# available for train/val. Overwrites train_pairs/val_pairs from the block
# above — this is the split the checkpoints in this notebook are trained
# with, matching 020_training.ipynb's active split.
pairs_mockups = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs_mockups,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Loss function

`attention_unet_nll` outputs two channels per pixel — `mu` (mean, same role as the deterministic models' single output) and `log_var` (log-variance, clipped to `[settings.NLL_LOG_VAR_MIN, settings.NLL_LOG_VAR_MAX]`) — trained with the Gaussian negative-log-likelihood:

```
loss = 0.5 * exp(-log_var) * (y_true - mu) ** 2 + 0.5 * log_var
```

The first term is an uncertainty-weighted squared error; the second penalises inflating `log_var` to trivially shrink the first. See `code-review.md` §7.6 for the full rationale and references (Nix & Weigend 1994; Kendall & Gal 2017; Seitzer et al. 2022).

Metrics (`mae`, `ssim`, `psnr`) are computed from the `mu` channel only (`scripts.metrics.Mu*Metric`), so they stay directly comparable to the deterministic architectures' metrics in `030_evaluation.ipynb`.

## 3. Train `attention_unet_nll`

Set `EPOCHS = 2` for a quick smoke test before committing to a full run.

In [ ]:
ARCH = "attention_unet_nll"
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test

model = get_model_nll(ARCH)
model = compile_model_nll(model, lr=settings.LEARNING_RATE)
model.summary(line_length=80)

callbacks = get_callbacks(
    ARCH,
    log_dir=settings.LOGS_DIR,
    model_dir=settings.MODELS_DIR,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

best_val_loss = min(history.history["val_loss"])
print(f"\nBest val_loss ({ARCH}): {best_val_loss:.4f}")

## 4. Training curves

In [ ]:
fig = plot_training_curves(history.history, title=f"Training history — {ARCH}")
plt.show()

## 5. Summary

Checkpoint saved to `models/attention_unet_nll/best_model.keras`.  
Run `tensorboard --logdir logs/` to inspect curves interactively.

In [ ]:
ckpt = settings.MODELS_DIR / ARCH / "best_model.keras"
status = "found" if ckpt.exists() else "MISSING"
print(f"{ARCH:<25}: {status}  ({ckpt})")